# GAP 1 – Notebook 03: Causal Analysis
**Causal graph learning + intervention simulation**

Covers:  
1. Causal graph structure (Domain-Knowledge DAG, 12 edges)  
2. Causal effects table (partial-correlation estimates)  
3. Refutation test results  
4. Intervention simulation visualisation  
5. Causal chain interpretation  

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

print('Libraries loaded')

## 1. Causal Graph Structure

In [ ]:
from causal_structure import load_causal_graph

G, method = load_causal_graph()
print(f'Method  : {method}')
print(f'Nodes   : {G.number_of_nodes()}')
print(f'Edges   : {G.number_of_edges()}')
print('\nEdges (sorted by |weight|):')
edges = [(u, v, d.get("weight", 0)) for u, v, d in G.edges(data=True)]
for u, v, w in sorted(edges, key=lambda x: abs(x[2]), reverse=True):
    print(f'  {u:28s} → {v:12s}  w={w:+.3f}')

In [ ]:
graph_img = '../results/figures/causal_graph.png'
if os.path.exists(graph_img):
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.imshow(mpimg.imread(graph_img))
    ax.axis('off')
    ax.set_title('Learned Causal DAG', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Run src/causal_structure.py first to generate the graph image.')

## 2. Causal Effects

In [ ]:
from causal_inference import load_causal_results

data   = load_causal_results()
effects = data['causal_effects']

rows = []
for t, r in effects.items():
    rows.append({
        'Treatment'  : t,
        'Effect'     : r['causal_effect'],
        'Method'     : r['method'],
        'Refutation' : 'PASS' if r.get('refutation_passed') else 'N/A',
    })

df_eff = pd.DataFrame(rows).sort_values('Effect', key=abs, ascending=False)
df_eff.style.background_gradient(subset=['Effect'], cmap='RdYlGn')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#e74c3c' if e > 0 else '#2ecc71' for e in df_eff['Effect']]
bars = ax.barh(df_eff['Treatment'], df_eff['Effect'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Causal Effect on P(sell)')
ax.set_title('Causal Effects on Investment Decision (Partial Correlation, OLS)', fontsize=12)
for bar in bars:
    w = bar.get_width()
    ax.text(w + (0.003 if w >= 0 else -0.003), bar.get_y() + bar.get_height()/2,
            f'{w:+.4f}', va='center', ha='left' if w >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Intervention Simulation

In [ ]:
inter = data['interventions']
rows2 = []
for k, v in inter.items():
    rows2.append({
        'Intervention' : k,
        'Description'  : v['description'],
        'P(sell) Before': f"{v['original_sell_prob']:.0%}",
        'P(sell) After' : f"{v['new_sell_prob']:.0%}",
        'Delta'         : f"{v['delta_sell_prob']:+.4f}",
        'Recommendation': v['recommendation'],
    })
pd.DataFrame(rows2)

In [ ]:
chart_path = '../results/figures/intervention_comparison.png'
if os.path.exists(chart_path):
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(mpimg.imread(chart_path))
    ax.axis('off')
    ax.set_title('Intervention Comparison Chart', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('Run src/causal_inference.py first to generate this chart.')

## 4. Causal Chain Interpretation

In [ ]:
print('=== CAUSAL INTERPRETATIONS ===')
for t, r in effects.items():
    print(f"\n{t}:")
    print(f"  {r['interpretation']}")